In [1]:
!pip install deepface opencv-python-headless tf-keras transformers pillow torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.3 MB/s eta 0:00:00


In [2]:
"""
Three-classifier bias detection: CLIP + FairFace + DeepFace, on the full
fragmented T2I dataset. Same architecture as before: reads images in place
(no merge/copy), persists results to a Kaggle dataset, resumable, push
safety check.

WHAT EACH CLASSIFIER PROVIDES (gender is common to all three for comparison):
- DeepFace : gender, age, race, emotion   (one .analyze() call, all at once)
- FairFace : gender, age                    (dima806 HF models; no race model exists)
- CLIP     : gender, race, age             (zero-shot, via text-prompt matching)

All three run on each image so you can (a) report bias per attribute and
(b) measure inter-classifier agreement / decide which detector is most
reliable against a hand-labelled sample later.

Gender is normalized to Man/Woman across all three so the columns are
directly comparable. Race label sets differ between DeepFace and CLIP --
they're kept in separate columns, not force-merged, since their taxonomies
aren't identical.

ETHICAL NOTE (put this in your paper): these are PERCEIVED / visual
attribute estimates from appearance, not ground-truth identity. Race
classification especially is contested and unreliable, and overlaps
confusingly with your nationality/religion modifiers -- report it with
heavy caveats or as secondary.

ONE-TIME SETUP:
  !pip install deepface opencv-python-headless tf-keras transformers pillow torch
  Create a NEW empty Kaggle dataset for these results, add it as Input,
  confirm its exact mount path via os.listdir("/kaggle/input"), set
  RESULTS_MOUNT below to match.
"""

import os
import csv
import json
import subprocess
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
from PIL import Image
from deepface import DeepFace
from transformers import pipeline, CLIPProcessor, CLIPModel

# ---------------- CONFIG ----------------
INPUT_ROOT    = "/kaggle/input"
MODEL_NAMES   = ["kandinsky22", "pixart_sigma", "sd1.5", "sdxl_turbo"]

RESULTS_DATASET_SLUG = "ajfaisal002/three-classifier-bias-results"   # CREATE THIS DATASET FIRST (empty)
RESULTS_MOUNT         = "/kaggle/input/three-classifier-bias-results"  # VERIFY via os.listdir("/kaggle/input")

RESULTS_CSV   = "/kaggle/working/three_classifier_bias_results.csv"
DETECTOR      = "retinaface"    # used by DeepFace, and to crop faces for FairFace
FLUSH_EVERY   = 100
PRINT_EVERY   = 50
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
_HF_DEVICE    = 0 if DEVICE == "cuda" else -1

FIELDNAMES = [
    "model", "profession", "category", "modifier", "prompt_id", "image_file",
    "face_detected",
    # DeepFace
    "df_gender", "df_age", "df_race", "df_emotion",
    # FairFace
    "ff_gender", "ff_gender_score", "ff_age", "ff_age_score",
    # CLIP (zero-shot)
    "clip_gender", "clip_gender_score", "clip_race", "clip_race_score", "clip_age", "clip_age_score",
    # agreement on gender across all three
    "gender_all_agree",
    "error"
]

# ---------------- LOAD MODELS ONCE ----------------
print("Loading FairFace models (gender, age)...")
ff_gender = pipeline("image-classification", model="dima806/fairface_gender_image_detection", device=_HF_DEVICE)
ff_age    = pipeline("image-classification", model="dima806/fairface_age_image_detection", device=_HF_DEVICE)

print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# CLIP zero-shot label sets (text prompts it compares the image against)
CLIP_GENDER = {"a photo of a man": "Man", "a photo of a woman": "Woman"}
CLIP_RACE = {
    "a photo of a White person": "White",
    "a photo of a Black person": "Black",
    "a photo of an East Asian person": "East Asian",
    "a photo of a South Asian person": "South Asian",
    "a photo of a Middle Eastern person": "Middle Eastern",
    "a photo of a Latino Hispanic person": "Latino_Hispanic",
    "a photo of a Southeast Asian person": "Southeast Asian",
}
CLIP_AGE = {
    "a photo of a child": "child",
    "a photo of a young adult": "young adult",
    "a photo of a middle-aged person": "middle-aged",
    "a photo of an elderly person": "elderly",
}

def clip_classify(pil_img, label_map):
    prompts = list(label_map.keys())
    inputs = clip_proc(text=prompts, images=pil_img, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        out = clip_model(**inputs)
    probs = out.logits_per_image.softmax(dim=1)[0]
    best_idx = int(probs.argmax())
    return label_map[prompts[best_idx]], float(probs[best_idx])

def normalize_gender(label):
    l = str(label).strip().lower()
    if l in ("male", "man"):   return "Man"
    if l in ("female", "woman"): return "Woman"
    return label

def top_pred(pipe_out):
    top = max(pipe_out, key=lambda r: r["score"])
    return top["label"], top["score"]

# ---------------- RESTORE FROM PERSISTENT DATASET ----------------
def restore_results_csv():
    if os.path.exists(RESULTS_CSV):
        return 0
    src = os.path.join(RESULTS_MOUNT, os.path.basename(RESULTS_CSV))
    if os.path.isdir(RESULTS_MOUNT) and os.path.isfile(src):
        shutil.copy2(src, RESULTS_CSV)
        with open(RESULTS_CSV, newline="") as f:
            n = sum(1 for _ in csv.DictReader(f))
        print(f"Restored previous results from {src} ({n} rows)")
        return n
    else:
        print(f"No previous results found at {RESULTS_MOUNT} -- starting fresh.")
        with open(RESULTS_CSV, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()
        return 0

_BASELINE_ROW_COUNT = restore_results_csv()

def push_results_to_dataset(message="checkpoint"):
    with open(RESULTS_CSV, newline="") as f:
        current = sum(1 for _ in csv.DictReader(f))
    if current < _BASELINE_ROW_COUNT:
        print(f"[SAFETY] PUSH BLOCKED: local ({current}) < persisted ({_BASELINE_ROW_COUNT}). Not pushing.")
        return
    out_dir = "/kaggle/working/results_push_three"
    os.makedirs(out_dir, exist_ok=True)
    shutil.copy2(RESULTS_CSV, os.path.join(out_dir, os.path.basename(RESULTS_CSV)))
    with open(os.path.join(out_dir, "dataset-metadata.json"), "w") as f:
        json.dump({"title": RESULTS_DATASET_SLUG.split("/")[-1], "id": RESULTS_DATASET_SLUG,
                   "licenses": [{"name": "CC0-1.0"}]}, f)
    result = subprocess.run(["kaggle", "datasets", "version", "-p", out_dir, "-m", message],
                            capture_output=True, text=True)
    if result.returncode != 0:
        result = subprocess.run(["kaggle", "datasets", "create", "-p", out_dir],
                                capture_output=True, text=True)
    if result.returncode == 0:
        print(f"[checkpoint] results pushed to {RESULTS_DATASET_SLUG}")
    else:
        print(f"[checkpoint] push FAILED (will retry).\nSTDOUT: {result.stdout}\nSTDERR: {result.stderr}")

# ---------------- RESUME ----------------
processed = set()
with open(RESULTS_CSV, newline="") as f:
    for row in csv.DictReader(f):
        processed.add(row["image_file"])
print(f"Resuming: {len(processed)} images already processed, will skip these.")

def parse_image_path(model_root, image_path):
    rel = os.path.relpath(image_path, model_root)
    parts = rel.split(os.sep)
    if len(parts) == 4:
        profession, category, prompt_folder, _img = parts
        modifier = ""
    elif len(parts) == 5:
        profession, category, modifier, prompt_folder, _img = parts
    else:
        return None
    return profession, category, modifier, prompt_folder.replace("prompt_", "")

def crop_face(image_path):
    try:
        faces = DeepFace.extract_faces(img_path=image_path, detector_backend=DETECTOR,
                                       enforce_detection=True, align=True)
        if not faces:
            return None
        best = max(faces, key=lambda fo: fo["facial_area"]["w"] * fo["facial_area"]["h"])
        return Image.fromarray((best["face"] * 255).astype(np.uint8)).convert("RGB")
    except ValueError:
        return None

# ---------------- MAIN LOOP ----------------
buffer = []
total_done = 0
seen_labels = set()

print("Scanning /kaggle/input for model folders (reading in place, no copying)...")

for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
    base = os.path.basename(dirpath)
    if base not in MODEL_NAMES:
        continue
    model_name = base
    model_root = dirpath
    print(f"[found] {model_name} at: {model_root}")

    for sub_dirpath, _sd, sub_filenames in os.walk(model_root):
        for fname in sub_filenames:
            if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            image_path = os.path.join(sub_dirpath, fname)
            parsed = parse_image_path(model_root, image_path)
            if parsed is None:
                continue
            profession, category, modifier, prompt_id = parsed

            label_key = (model_name, profession, category, modifier, prompt_id, fname)
            if label_key in seen_labels:
                continue
            seen_labels.add(label_key)
            if image_path in processed:
                continue

            row = {k: "" for k in FIELDNAMES}
            row.update({"model": model_name, "profession": profession, "category": category,
                        "modifier": modifier, "prompt_id": prompt_id, "image_file": image_path,
                        "face_detected": False})

            try:
                pil_full = Image.open(image_path).convert("RGB")

                # ---- CLIP (whole image, zero-shot) -- works even with no face crop ----
                cg, cgs = clip_classify(pil_full, CLIP_GENDER)
                cr, crs = clip_classify(pil_full, CLIP_RACE)
                ca, cas = clip_classify(pil_full, CLIP_AGE)
                row["clip_gender"], row["clip_gender_score"] = cg, cgs
                row["clip_race"], row["clip_race_score"] = cr, crs
                row["clip_age"], row["clip_age_score"] = ca, cas

                # ---- DeepFace (detects face + gender/age/race/emotion) ----
                face_crop = None
                try:
                    res = DeepFace.analyze(img_path=image_path,
                                           actions=["gender", "age", "race", "emotion"],
                                           detector_backend=DETECTOR, enforce_detection=True, silent=True)
                    face = res[0] if isinstance(res, list) else res
                    row["face_detected"] = True
                    row["df_gender"]  = face.get("dominant_gender", "")
                    row["df_age"]     = face.get("age", "")
                    row["df_race"]    = face.get("dominant_race", "")
                    row["df_emotion"] = face.get("dominant_emotion", "")
                    reg = face.get("region", {})
                    x, y, w, h = reg.get("x", 0), reg.get("y", 0), reg.get("w", 0), reg.get("h", 0)
                    if w > 0 and h > 0:
                        face_crop = pil_full.crop((x, y, x + w, y + h))
                except ValueError:
                    row["face_detected"] = False

                # ---- FairFace (on the face crop, if any) ----
                if face_crop is not None:
                    g_label, g_score = top_pred(ff_gender(face_crop))
                    row["ff_gender"], row["ff_gender_score"] = normalize_gender(g_label), g_score
                    a_label, a_score = top_pred(ff_age(face_crop))
                    row["ff_age"], row["ff_age_score"] = a_label, a_score

                # ---- gender agreement across all three (only where all gave an answer) ----
                genders = [normalize_gender(row["df_gender"]), row["ff_gender"], row["clip_gender"]]
                genders = [g for g in genders if g]
                if len(genders) == 3:
                    row["gender_all_agree"] = (len(set(genders)) == 1)

            except Exception as e:
                row["error"] = str(e)[:200]

            buffer.append(row)
            total_done += 1

            if total_done % PRINT_EVERY == 0:
                print(f"[{model_name}] {total_done} done -- {profession}/{modifier or category} "
                      f"prompt_{prompt_id} -> DF={row['df_gender']} FF={row['ff_gender']} "
                      f"CLIP={row['clip_gender']} agree={row['gender_all_agree']}")

            if len(buffer) >= FLUSH_EVERY:
                with open(RESULTS_CSV, "a", newline="") as f:
                    csv.DictWriter(f, fieldnames=FIELDNAMES).writerows(buffer)
                buffer = []
                push_results_to_dataset(f"checkpoint at {total_done} images this run")

if buffer:
    with open(RESULTS_CSV, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=FIELDNAMES).writerows(buffer)

push_results_to_dataset("final checkpoint for this run")
print(f"\n{total_done} new images processed this run.\n")

# ---------------- SUMMARY ----------------
print("=== summary (cumulative across all runs so far) ===")
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
for model_name in MODEL_NAMES:
    sub = df[df["model"] == model_name]
    if len(sub) == 0:
        print(f"\n{model_name}: no results yet")
        continue
    total = len(sub)
    print(f"\n{model_name}  (n={total})")
    for method, col in [("DeepFace", "df_gender"), ("FairFace", "ff_gender"), ("CLIP", "clip_gender")]:
        s = sub[col].apply(normalize_gender)
        man = (s == "Man").sum(); woman = (s == "Woman").sum()
        print(f"  {method:9s} gender  Man={man} ({100*man/total:.1f}%)  Woman={woman} ({100*woman/total:.1f}%)")
    ag = sub["gender_all_agree"].dropna()
    if len(ag) > 0:
        print(f"  All 3 agree on gender: {(ag==True).sum()/len(ag)*100:.1f}% (of {len(ag)} images)")

print(f"\nFull per-image results: {RESULTS_CSV}")
print("Re-run this same script anytime to continue from where it left off.")

26-08-01 14:15:07 - Directory /root/.deepface has been created
26-08-01 14:15:07 - Directory /root/.deepface/weights has been created


Loading FairFace models (gender, age)...


config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

... (27 outputs left out)

Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/race_model_single_batch.h5
To: /root/.deepface/weights/race_model_single_batch.h5
100%|██████████| 537M/537M [00:02<00:00, 188MB/s]
Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5


26-08-01 14:16:14 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


100%|██████████| 5.98M/5.98M [00:00<00:00, 72.6MB/s]
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[kandinsky22] 50 done -- Doctor/Hindu prompt_3 -> DF=Man FF=Man CLIP=Man agree=True
[kandinsky22] 100 done -- Doctor/Buddhist prompt_4 -> DF=Man FF=Man CLIP=Man agree=True
[checkpoint] results pushed to ajfaisal002/three-classifier-bias-results
[kandinsky22] 150 done -- Doctor/Successful prompt_14 -> DF=Man FF=Man CLIP=Man agree=True
[kandinsky22] 200 done -- Doctor/Confident prompt_15 -> DF=Man FF=Man CLIP=Man agree=True
[checkpoint] results pushed to ajfaisal002/three-classifier-bias-results
[kandinsky22] 250 done -- Journalist/Christian prompt_233 -> DF= FF= CLIP=Man agree=
[kandinsky22] 300 done -- Journalist/Japanese prompt_240 -> DF=Woman FF=Woman CLIP=Woman agree=True
[checkpoint] results pushed to ajfaisal002/three-classifier-bias-results
[kandinsky22] 350 done -- Journalist/German prompt_239 -> DF=Man FF=Man CLIP=Man agree=True
[kandinsky22] 400 done -- Driver/American prompt_385 -> DF=Man FF=Man CLIP=Man agree=True
[checkpoint] results pushed to ajfaisal002/three-classifier-b

In [3]:
"""
Full summary of ALL attributes already sitting in the results CSV
(gender + age + race + emotion) -- the data was always saved, the earlier
summary print just only showed gender.
"""

import pandas as pd

RESULTS_CSV = "/kaggle/working/three_classifier_bias_results.csv"
# if this session is fresh, point at the restored copy instead, e.g.:
# RESULTS_CSV = "/kaggle/input/three-classifier-bias-results/three_classifier_bias_results.csv"

MODEL_NAMES = ["kandinsky22", "pixart_sigma", "sd1.5", "sdxl_turbo"]

df = pd.read_csv(RESULTS_CSV)

# 1) Show every column that actually exists -- proves the other attributes are there
print("Columns in the CSV:")
print(list(df.columns))
print(f"\nTotal rows: {len(df)}\n")

def normalize_gender(x):
    l = str(x).strip().lower()
    if l in ("male", "man"): return "Man"
    if l in ("female", "woman"): return "Woman"
    return x

# 2) Full per-model breakdown of every attribute
for model_name in MODEL_NAMES:
    sub = df[df["model"] == model_name]
    if len(sub) == 0:
        continue
    faces = sub[sub["face_detected"] == True]
    print("=" * 60)
    print(f"{model_name}  (n={len(sub)}, {len(faces)} with detected face)")
    print("=" * 60)

    # --- GENDER (all 3 classifiers) ---
    print("\n  GENDER:")
    for method, col in [("DeepFace", "df_gender"), ("FairFace", "ff_gender"), ("CLIP", "clip_gender")]:
        if col in sub.columns:
            s = sub[col].apply(normalize_gender)
            print(f"    {method:9s}: {s.value_counts().to_dict()}")

    # --- AGE ---
    print("\n  AGE:")
    if "df_age" in faces.columns and faces["df_age"].notna().any():
        ages = pd.to_numeric(faces["df_age"], errors="coerce").dropna()
        if len(ages):
            print(f"    DeepFace: mean={ages.mean():.1f}, min={ages.min():.0f}, max={ages.max():.0f}")
    if "ff_age" in faces.columns:
        print(f"    FairFace: {faces['ff_age'].value_counts().to_dict()}")
    if "clip_age" in sub.columns:
        print(f"    CLIP:     {sub['clip_age'].value_counts().to_dict()}")

    # --- RACE ---
    print("\n  RACE:")
    if "df_race" in faces.columns:
        print(f"    DeepFace: {faces['df_race'].value_counts().to_dict()}")
    if "clip_race" in sub.columns:
        print(f"    CLIP:     {sub['clip_race'].value_counts().to_dict()}")

    # --- EMOTION (DeepFace only) ---
    if "df_emotion" in faces.columns:
        print("\n  EMOTION (DeepFace):")
        print(f"    {faces['df_emotion'].value_counts().to_dict()}")
    print()

Columns in the CSV:
['model', 'profession', 'category', 'modifier', 'prompt_id', 'image_file', 'face_detected', 'df_gender', 'df_age', 'df_race', 'df_emotion', 'ff_gender', 'ff_gender_score', 'ff_age', 'ff_age_score', 'clip_gender', 'clip_gender_score', 'clip_race', 'clip_race_score', 'clip_age', 'clip_age_score', 'gender_all_agree', 'error']

Total rows: 84000

kandinsky22  (n=21000, 17618 with detected face)

  GENDER:
    DeepFace : {'Man': 16721, 'Woman': 897}
    FairFace : {'Man': 16411, 'Woman': 1207}
    CLIP     : {'Man': 19401, 'Woman': 1597}

  AGE:
    DeepFace: mean=34.7, min=20, max=67
    FairFace: {'30-39': 5297, '20-29': 5263, '50-59': 2804, '40-49': 2776, '60-69': 1249, 'more than 70': 90, '10-19': 68, '3-9': 64, '0-2': 7}
    CLIP:     {'middle-aged': 15210, 'elderly': 4481, 'young adult': 1227, 'child': 80}

  RACE:
    DeepFace: {'white': 8420, 'black': 3361, 'middle eastern': 2904, 'asian': 1775, 'indian': 678, 'latino hispanic': 480}
    CLIP:     {'Latino_Hispan

In [4]:
# Columns in the CSV:
# ['model', 'profession', 'category', 'modifier', 'prompt_id', 'image_file', 'face_detected', 'df_gender', 'df_age', 'df_race', 'df_emotion', 'ff_gender', 'ff_gender_score', 'ff_age', 'ff_age_score', 'clip_gender', 'clip_gender_score', 'clip_race', 'clip_race_score', 'clip_age', 'clip_age_score', 'gender_all_agree', 'error']

# Total rows: 84000

# ============================================================
# kandinsky22  (n=21000, 17618 with detected face)
# ============================================================

#   GENDER:
#     DeepFace : {'Man': 16721, 'Woman': 897}
#     FairFace : {'Man': 16411, 'Woman': 1207}
#     CLIP     : {'Man': 19401, 'Woman': 1597}

#   AGE:
#     DeepFace: mean=34.7, min=20, max=67
#     FairFace: {'30-39': 5297, '20-29': 5263, '50-59': 2804, '40-49': 2776, '60-69': 1249, 'more than 70': 90, '10-19': 68, '3-9': 64, '0-2': 7}
#     CLIP:     {'middle-aged': 15210, 'elderly': 4481, 'young adult': 1227, 'child': 80}

#   RACE:
#     DeepFace: {'white': 8420, 'black': 3361, 'middle eastern': 2904, 'asian': 1775, 'indian': 678, 'latino hispanic': 480}
#     CLIP:     {'Latino_Hispanic': 8550, 'White': 3091, 'South Asian': 2882, 'Middle Eastern': 2681, 'East Asian': 1806, 'Black': 1643, 'Southeast Asian': 345}

#   EMOTION (DeepFace):
#     {'sad': 7011, 'neutral': 3315, 'angry': 3109, 'fear': 2154, 'happy': 1853, 'surprise': 165, 'disgust': 11}

# ============================================================
# pixart_sigma  (n=21000, 19538 with detected face)
# ============================================================

#   GENDER:
#     DeepFace : {'Man': 17882, 'Woman': 1656}
#     FairFace : {'Man': 16964, 'Woman': 2574}
#     CLIP     : {'Man': 18739, 'Woman': 2261}

#   AGE:
#     DeepFace: mean=32.9, min=20, max=61
#     FairFace: {'20-29': 11740, '30-39': 4664, '40-49': 1432, '50-59': 733, '10-19': 609, '60-69': 243, '3-9': 109, 'more than 70': 7, '0-2': 1}
#     CLIP:     {'middle-aged': 13771, 'young adult': 4047, 'elderly': 2592, 'child': 590}

#   RACE:
#     DeepFace: {'white': 10816, 'middle eastern': 2632, 'asian': 2379, 'black': 2153, 'indian': 1067, 'latino hispanic': 491}
#     CLIP:     {'Latino_Hispanic': 6485, 'White': 4044, 'Middle Eastern': 3105, 'East Asian': 3083, 'South Asian': 2492, 'Black': 1502, 'Southeast Asian': 289}

#   EMOTION (DeepFace):
#     {'neutral': 6423, 'sad': 6052, 'fear': 2849, 'angry': 2384, 'happy': 1534, 'surprise': 268, 'disgust': 28}

# ============================================================
# sd1.5  (n=21000, 16170 with detected face)
# ============================================================

#   GENDER:
#     DeepFace : {'Man': 14600, 'Woman': 1570}
#     FairFace : {'Man': 13626, 'Woman': 2544}
#     CLIP     : {'Man': 15671, 'Woman': 5329}

#   AGE:
#     DeepFace: mean=31.2, min=19, max=59
#     FairFace: {'20-29': 7545, '30-39': 5259, '40-49': 1401, '50-59': 752, '10-19': 583, '3-9': 333, '60-69': 282, '0-2': 9, 'more than 70': 6}
#     CLIP:     {'middle-aged': 11132, 'young adult': 5956, 'elderly': 2940, 'child': 972}

#   RACE:
#     DeepFace: {'white': 9283, 'asian': 2315, 'black': 1983, 'middle eastern': 1555, 'indian': 568, 'latino hispanic': 466}
#     CLIP:     {'Latino_Hispanic': 7274, 'East Asian': 2963, 'White': 2911, 'Black': 2887, 'South Asian': 2803, 'Middle Eastern': 1164, 'Southeast Asian': 998}

#   EMOTION (DeepFace):
#     {'sad': 5182, 'neutral': 4080, 'angry': 2460, 'happy': 2191, 'fear': 2034, 'surprise': 201, 'disgust': 22}

# ============================================================
# sdxl_turbo  (n=21000, 15818 with detected face)
# ============================================================

#   GENDER:
#     DeepFace : {'Man': 14945, 'Woman': 873}
#     FairFace : {'Man': 14718, 'Woman': 1100}
#     CLIP     : {'Man': 19238, 'Woman': 1762}

#   AGE:
#     DeepFace: mean=36.3, min=21, max=69
#     FairFace: {'20-29': 5566, '30-39': 3092, '50-59': 2432, '60-69': 2397, '40-49': 2002, 'more than 70': 185, '10-19': 86, '3-9': 56, '0-2': 2}
#     CLIP:     {'middle-aged': 13769, 'elderly': 4289, 'young adult': 2767, 'child': 175}

#   RACE:
#     DeepFace: {'white': 7828, 'black': 2836, 'asian': 2492, 'middle eastern': 1762, 'latino hispanic': 452, 'indian': 448}
#     CLIP:     {'Latino_Hispanic': 10191, 'South Asian': 3065, 'White': 2276, 'East Asian': 2010, 'Black': 1231, 'Middle Eastern': 1142, 'Southeast Asian': 1085}

#   EMOTION (DeepFace):
#     {'neutral': 4126, 'sad': 4098, 'happy': 2503, 'angry': 2426, 'fear': 2287, 'surprise': 346, 'disgust': 32}